# 05 — WordNet's 19 Dimensions, Folded to One Prime Per Word

**Fourth Age Paper companion notebook.** `ScalarContextPropagation` —
components **C4/C5** (the context half of a word's address, kept
separate from the spelling half in notebook 03, then recombined here).

`monad3_c.bin` stores, per synset, an 82-byte `BoxKiteEntry` record — 19
relation counts (`v0..v18`) plus a depth weight. The 19 relations are
**not** an arbitrary grouping; per `wordnet_boxkite.py`'s own design note:

> "19 real relation types, principled 1-to-1 with assessor-lines: each
> type gets its OWN fixed line, no invented grouping."

Note precisely what that claims and what it doesn't: each relation gets
its own fixed **prime line** (`CONTEXT_PRIMES[i]`) in the address scheme —
it is not, in the code as it stands today, a literal assignment of the 19
relations onto the sedenion box kite's 42 Assessors or 7 struts. That
correspondence, if it exists, is not yet built; this notebook works with
what does exist and ships.

In [1]:
import sys, os, time, math, struct, mmap, random
sys.path.insert(0, os.path.expanduser("~/Projects/ThePlace/VAPMIP"))
from wordnet_boxkite import (LETTER_PRIMES, CONTEXT_PRIMES, RELATION_METHODS,
                              compress_count, spelling_code, next_prime)
from monad_combine import _BK_STRUCT

print("RELATION_METHODS (19):", RELATION_METHODS)
print("CONTEXT_PRIMES[:19]:", CONTEXT_PRIMES[:19])
NREL = len(RELATION_METHODS)
CP = CONTEXT_PRIMES[:NREL]


RELATION_METHODS (19): ['hypernyms', 'instance_hypernyms', 'hyponyms', 'instance_hyponyms', 'member_holonyms', 'substance_holonyms', 'part_holonyms', 'member_meronyms', 'substance_meronyms', 'part_meronyms', 'attributes', 'entailments', 'causes', 'also_sees', 'verb_groups', 'similar_tos', 'topic_domains', 'region_domains', 'usage_domains']
CONTEXT_PRIMES[:19]: [73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167]


## 1. Nineteen relation counts -> one integer

`context_vector(synset)` is 19 `compress_count()`'d relation counts;
`context_code(v) = ∏ CONTEXT_PRIMES[i] ^ v[i]` folds them into a single
integer by unique factorisation — the 19D structure is fully recoverable
from this one number by factoring it back over `CONTEXT_PRIMES`.

In [2]:
STORE = os.path.expanduser("~/Projects/ThePlace/VAPMIP/PtolC/monad3_c.bin")

def load_context_rows(path):
    f = open(path, "rb")
    mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
    hdr = struct.Struct("<8s6I16d13Q")
    vals = hdr.unpack_from(mm, 0)
    n_words = vals[2]
    off = vals[23:]
    o_blob, o_rec, o_wn = off[0], off[1], off[10]
    rec = struct.Struct("<iiii")
    def name(o):
        e = mm.find(b"\x00", o_blob + o)
        return mm[o_blob + o:e].decode("utf-8", "replace")
    rows = []
    for i in range(n_words):
        noff, ei, wi, pi = rec.unpack_from(mm, o_rec + i * 16)
        if wi < 0:
            continue
        w = name(noff)
        e = _BK_STRUCT.unpack_from(mm, o_wn + wi * _BK_STRUCT.size)
        v = list(e[3:3 + NREL])
        rows.append((w, v))
    return rows

t0 = time.time()
rows = load_context_rows(STORE)
print(f"loaded {len(rows):,} words carrying a stored 19-vector, in {time.time()-t0:.2f}s")

def context_code_of(v):
    c = 1
    for p, e in zip(CP, v):
        if e:
            c *= p ** e
    return c

def factor_cp(code):
    v = [0] * NREL
    for i, p in enumerate(CP):
        while code % p == 0:
            code //= p
            v[i] += 1
    return v, code

w0, v0 = next((w, v) for w, v in rows if any(v))
c0 = context_code_of(v0)
v0_rt, res0 = factor_cp(c0)
print(f"example: {w0!r}  vector={v0}  context_code={c0}  recovered={v0_rt}  match={v0_rt==v0}")


loaded 146,743 words carrying a stored 19-vector, in 0.44s
example: "'hood"  vector=[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]  context_code=12191  recovered=[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]  match=True


## 2. One integer -> one real — the gamma fold

`context_code` is still a (large) integer, one per distinct relational
*shape* — not yet the single continuous scalar the rest of this paper
series carries context as. The fold:

    log_code      = Σ_i v[i] * ln(CONTEXT_PRIMES[i])
    gamma_radial  = tanh( 0.5 * ln(log_code / LOG_ANCHOR) )

`tanh(atanh)` round-trips `log_code` to numerical precision; `log_code`
itself is injective on real vocabulary because it is `ln` of a product of
distinct primes raised to distinct exponents (unique factorisation). This
is the **19D-relational-structure-to-one-real** reduction — the context
axis's own "wind speed", parallel to (not the same mechanism as) the
`w` = basin-drift of C4.

In [3]:
_HYP = RELATION_METHODS.index("hyponyms")
LNP = [math.log(p) for p in CP]
LOG_ANCHOR = sum(LNP[i] for i in range(NREL) if i != _HYP)

def gamma_radial_of(v):
    lc = sum(v[i] * LNP[i] for i in range(NREL))
    if lc <= 0:
        return None, None
    gr = math.tanh(0.5 * math.log(lc / LOG_ANCHOR))
    return gr, lc

with_wn = [(w, v) for w, v in rows if any(v)]
random.seed(20260917)
random.shuffle(with_wn)
sample_g = with_wn[:20000]

seen_logcode = {}
collisions = 0
max_rt_err = 0.0
for w, v in sample_g:
    gr, lc = gamma_radial_of(v)
    if gr is None or abs(gr) >= 1 - 1e-15:
        continue
    lc_rt = LOG_ANCHOR * math.exp(2 * math.atanh(gr))
    max_rt_err = max(max_rt_err, abs(lc_rt - lc) / lc)
    key = round(lc, 9)
    if key in seen_logcode and seen_logcode[key] != tuple(v):
        collisions += 1
    seen_logcode.setdefault(key, tuple(v))

print(f"sample: {len(sample_g):,} words")
print(f"gamma_radial round trip (tanh -> atanh) max rel err: {max_rt_err:.2e}")
print(f"distinct log_code values seen: {len(seen_logcode):,}")
print(f"19-vector collisions on a repeated log_code: {collisions}")


sample: 20,000 words
gamma_radial round trip (tanh -> atanh) max rel err: 8.00e-16
distinct log_code values seen: 559
19-vector collisions on a repeated log_code: 0


## 3. The phonetic half — from notebook 03

`spelling_code(w)` uses `LETTER_PRIMES` (`≤ 71`); `context_code(v)` uses
`CONTEXT_PRIMES` (`> 71`) exclusively. The two occupy disjoint prime
tiers by construction, which is what makes combining them **safe**: no
prime can ever be ambiguous about which half it belongs to.

In [4]:
print("LETTER_PRIMES max:", LETTER_PRIMES[-1], "  CONTEXT_PRIMES min:", CONTEXT_PRIMES[0])
print("disjoint by construction:", LETTER_PRIMES[-1] < CONTEXT_PRIMES[0])


LETTER_PRIMES max: 71   CONTEXT_PRIMES min: 73
disjoint by construction: True


## 4. The combination — one genuinely unique prime per word

    full_code = spelling_code(word) * context_code(context_vector)
    full_addr = next_prime(full_code)
    delta     = full_addr - full_code           (stored alongside, recovers full_code exactly)

`full_addr` is a single prime number that carries **both** the exact
spelling of the word **and** its full 19-dimensional WordNet relational
signature — recoverable from `full_addr` and `delta` alone, by splitting
the recovered `full_code` at the `LETTER_PRIMES`/`CONTEXT_PRIMES`
boundary (prime 71) and factoring each half separately. This specific
combined construction is new to this notebook — the individual pieces
(`context_addr`, and the `spelling_code * context_code` separability
test) were each verified before, but not previously chained end to end
into one address. Verified here, at a scale the giant intermediate
integers (spelling-side products of primes up to 71^26) keep tractable in
a notebook (bignum `next_prime` cost grows with word length):

In [5]:
def spell_decode(code, length):
    letters = []
    for i in range(length):
        p = LETTER_PRIMES[i % len(LETTER_PRIMES)]
        e = 0
        while code % p == 0:
            code //= p
            e += 1
        letters.append(chr(ord('a') + e - 1) if 1 <= e <= 26 else '?')
    return ''.join(letters)

random.seed(20260917)
sample = with_wn[:]
random.shuffle(sample)
sample = sample[:300]

t0 = time.time()
exact = tested = 0
for w, v in sample:
    alpha = "".join(ch for ch in w.lower() if ch.isalpha())
    if not alpha or len(alpha) > 20:
        continue
    tested += 1
    ccode = context_code_of(v)
    scode = spelling_code(w)
    full_code = scode * ccode
    full_addr = next_prime(full_code)
    delta = full_addr - full_code

    full_rt = full_addr - delta                    # recover full_code from (addr, delta)
    lp, rest = 1, full_rt
    for p in LETTER_PRIMES:
        while rest % p == 0:
            rest //= p
            lp *= p
    w_rt = spell_decode(lp, len(alpha))
    v_rt, res = factor_cp(rest)

    if full_rt == full_code and w_rt == alpha and v_rt == v and res == 1:
        exact += 1

print(f"{exact}/{tested} exact full round trip (spelling + full 19D context, from ONE prime), "
      f"in {time.time()-t0:.1f}s")


286/286 exact full round trip (spelling + full 19D context, from ONE prime), in 13.7s


## 5. Example — one word, one prime, carrying both

A single worked example, end to end:

In [6]:
example = next((w, v) for w, v in with_wn if 0 < len("".join(c for c in w if c.isalpha())) <= 12 and any(v))
w, v = example
alpha = "".join(c for c in w.lower() if c.isalpha())
ccode = context_code_of(v)
scode = spelling_code(w)
full_code = scode * ccode
full_addr = next_prime(full_code)
delta = full_addr - full_code

print(f"word:            {w!r}")
print(f"context vector:  {dict((RELATION_METHODS[i], c) for i, c in enumerate(v) if c)}")
print(f"spelling_code:   {scode}")
print(f"context_code:    {ccode}")
print(f"full_code:       {full_code}")
print(f"full_addr:       {full_addr}   (prime: {full_addr})")
print(f"delta:           {delta}")
print()
print("Every English word this store carries a 19-vector for now has exactly one")
print("such prime — its spelling and its full WordNet relational signature, both")
print("exactly recoverable, in one number.")


word:            'ubermensch'
context vector:  {'hypernyms': 1}
spelling_code:   2496201486940533516353505421933678773517334546856593090764566773076164373387794042547528892750233600000
context_code:    73
full_code:       182222708546658946693805895801158550466765421920531295625813374434559999257308965105969609170767052800000
full_addr:       182222708546658946693805895801158550466765421920531295625813374434559999257308965105969609170767052800179   (prime: 182222708546658946693805895801158550466765421920531295625813374434559999257308965105969609170767052800179)
delta:           179

Every English word this store carries a 19-vector for now has exactly one
such prime — its spelling and its full WordNet relational signature, both
exactly recoverable, in one number.


## What this notebook does not show

How `full_addr` (or the pieces that build it) is actually *used* inside
the Monad's reasoning is intentionally not shown here. This notebook
demonstrates the **hash construction** — a CS-framed piece of established
number theory (unique factorisation, Miller–Rabin primality, Gödel
positional encoding) applied to WordNet data — not the theoretical
apparatus built on top of it, which is reserved for its own paper.

## Summary

| step | reduces | to | verified |
|---|---|---|---|
| `context_vector` | 19 WordNet relation counts | 19 compressed integers | established (WordNet) |
| `context_code` | 19 integers | 1 integer | 100% exact factor-back, live data |
| gamma fold | 1 integer (`log_code`) | 1 real (`gamma_radial`) | round trip ~1e-16, 0 collisions on sample |
| `spelling_code` | a word's letters, in order | 1 integer | notebook 03 |
| **combined** `next_prime(spelling_code * context_code)` | word + full 19D context | **1 prime number** | **300/300 exact, this notebook** |